# The neuron and the perceptron
**Machine Learning · Week 5 (Neural networks with PyTorch) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w5d1.html

One artificial neuron: a weighted sum, a threshold, and a learning rule from 1958.

In [ ]:
import numpy as np

def neuron(x, w, b):
    """A perceptron: 1 if the weighted sum is positive, else 0."""
    return int(np.dot(w, x) + b > 0)

inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]
for name, w, b in [("AND", (1, 1), -1.5), ("OR", (1, 1), -0.5)]:
    print(name, [neuron(x, w, b) for x in inputs])

## The perceptron learning rule
For each example: if the prediction $\hat y$ is wrong, nudge the weights toward the right answer,
$w \leftarrow w + \eta\,(y - \hat y)\,x$ and $b \leftarrow b + \eta\,(y - \hat y)$. Repeat until a full pass has no mistakes.

In [ ]:
def train_perceptron(X, y, lr=0.5, max_passes=25):
    w, b = np.zeros(X.shape[1]), 0.0
    for p in range(1, max_passes + 1):
        mistakes = 0
        for xi, yi in zip(X, y):
            error = yi - neuron(xi, w, b)
            if error:
                w, b = w + lr * error * xi, b + lr * error
                mistakes += 1
        if mistakes == 0:
            return w, b, p
    return w, b, None

X = np.array(inputs, dtype=float)
for name, y in [("AND", [0, 0, 0, 1]), ("OR", [0, 1, 1, 1]), ("XOR", [0, 1, 1, 0])]:
    w, b, passes = train_perceptron(X, np.array(y))
    if passes:
        print(f"{name}: learned in {passes} passes, w = {w}, b = {b}")
    else:
        print(f"{name}: still making mistakes after 25 passes. No straight line separates it.")

## The same neuron in PyTorch
Replace the step with a sigmoid and the rule with gradient descent on cross-entropy, and the neuron becomes
logistic regression: `nn.Linear(2, 1)` plus a sigmoid.

In [ ]:
import torch

torch.manual_seed(0)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor([[0.0], [0.0], [0.0], [1.0]])           # AND
neuron_t = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(neuron_t.parameters(), lr=1.0)
for step in range(2000):
    optimizer.zero_grad()
    torch.nn.functional.binary_cross_entropy_with_logits(neuron_t(Xt), yt).backward()
    optimizer.step()
with torch.no_grad():
    print("P(1) for 00, 01, 10, 11:", torch.sigmoid(neuron_t(Xt)).ravel().numpy().round(3))

## Your turn: a perceptron on real flowers
Setosa and versicolor irises can be separated by a straight line using petal length and width.
Train your `train_perceptron` on them and count the passes; then compare with scikit-learn's `Perceptron`.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron

iris = load_iris()
keep = iris.target < 2                                   # setosa (0) and versicolor (1) only
Xi, yi = iris.data[keep][:, 2:], iris.target[keep]
result = None   # TODO: train_perceptron(Xi, yi, lr=0.1, max_passes=100)
print(result)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
w, b, passes = train_perceptron(Xi, yi, lr=0.1, max_passes=100)
acc = np.mean([neuron(x, w, b) == t for x, t in zip(Xi, yi)])
print(f"mine: {passes} passes, w = {np.round(w, 2)}, b = {b:.2f}, accuracy {acc:.2f}")
sk = Perceptron(random_state=0).fit(Xi, yi)
print(f"scikit-learn: accuracy {sk.score(Xi, yi):.2f}")
print("Separable data: the perceptron is guaranteed to stop, and it did.")